In [4]:
import gradio as gr
import subprocess
import google.generativeai as genai
import requests
import os
from dotenv import load_dotenv

# === CẤU HÌNH ===
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OLLAMA_MODEL = "mistral"  # Hoặc model khác bạn có trong Ollama

# === KẾT NỐI GEMINI ===
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

# === FUNCTION: GEMINI ===
def ask_gemini(prompt):
    try:
        response = gemini_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"[Gemini Error] {str(e)}"

# === FUNCTION: OLLAMA ===
def ask_ollama(prompt):
    try:
        response = requests.post("http://localhost:11434/api/generate", json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False
        })
        return response.json().get("response", "[No Response]")
    except Exception as e:
        return f"[Ollama Error] {str(e)}"

# === FUNCTION: SO SÁNH 2 MÔ HÌNH ===
def compare_models(task_type, user_input):
    if task_type == "Code Generation":
        prompt = f"Viết code cho yêu cầu sau bằng Python:\n{user_input}"
    else:
        prompt = f"Xử lý tác vụ kinh doanh sau bằng văn phong chuyên nghiệp:\n{user_input}"

    gemini_result = ask_gemini(prompt)
    ollama_result = ask_ollama(prompt)

    return gemini_result, ollama_result

# === UI ===
gr.Interface(
    fn=compare_models,
    inputs=[
        gr.Dropdown(["Code Generation", "Business Tasks"], label="Chọn loại nhiệm vụ"),
        gr.Textbox(lines=5, label="Yêu cầu (task)")
    ],
    outputs=[
        gr.Textbox(label="Gemini Kết quả"),
        gr.Textbox(label="Ollama Kết quả")
    ],
    title="🔍 LLM Showdown: Gemini vs Ollama",
    description="So sánh hai mô hình LLM với hai loại nhiệm vụ: sinh mã lập trình và tác vụ kinh doanh."
).launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
